# Music Flamingo Predictions

### Packages download and import

In [ ]:
# Install the needed packages
! pip install -r ../requierments.txt

In [ ]:
# Import packages
import random
import re
import pandas as pd
from tqdm import tqdm
from transformers import AudioFlamingo3ForConditionalGeneration, AutoProcessor
from peft import PeftModel
import torch

### Define the device and the model

To load the model directly from Huggingface change the _model_id_ variable to :

`model_id = 'nvidia/music-flamingo-hf'`


In [13]:
device = "cuda"
model_id = "./model-training/music-flamingo"

Base model :

In [ ]:
# BASE MODEL
model = AudioFlamingo3ForConditionalGeneration.from_pretrained(model_id,
                                                               dtype=torch.bfloat16,
                                                               device_map=device)
processor = AutoProcessor.from_pretrained(model_id)

Fine-tuned model :

In [ ]:
# FINETUNED MODEL
base_model = AudioFlamingo3ForConditionalGeneration.from_pretrained(model_id, 
                                                                    dtype=torch.bfloat16,
                                                                      device_map=device)
model = PeftModel.from_pretrained(base_model, "content/MF-fine-tuned")
processor = AutoProcessor.from_pretrained("/content/MF-fine-tuned")

### Predictions

#### Test Data and emotions definition

In [15]:
# Define test dataframe
recoded_data_path = '../data/test.csv'
df = pd.read_csv(recoded_data_path)
df.head()

,Unnamed: 0,artist,title,Wonder,Transcendence,Nostalgia,Tenderness,Peacefulness,Joy,Power,Tension,Sadness,Spotify ID,Youtube ID,genre,Path
0,457,Sugababes,Push The Button,5,2,4,2,2,5,5,3,1,3EDS89HeAhHiZKlljciK0a,Ugkxu8HzYufqmVieUjWOKqkoQqrrFpFbRnh7,pop,tracks/Ugkxu8HzYufqmVieUjWOKqkoQqrrFpFbRnh7.mp3
1,148,Lacrimas Profundere,The Crown of Leaving,3,3,3,2,3,2,3,4,3,7qNpAPSfPiLQ47cez2IpeW,NaN,rock,tracks/7qNpAPSfPiLQ47cez2IpeW.mp3
2,742,The Cinematic Orchestra,Time and Space,3,3,2,3,5,1,1,3,5,6qqNMdHr1hcyF4amMDP5Sf,NaN,electronic,tracks/6qqNMdHr1hcyF4amMDP5Sf.mp3
3,526,The Beach Boys,Good Vibrations,4,2,4,2,4,5,3,1,1,6aU6a9tdn2vHhnPGlboFZX,UgkxoWtHfVqF7dqNB3LH94zBUkV0DScEs0Gm,pop,tracks/UgkxoWtHfVqF7dqNB3LH94zBUkV0DScEs0Gm.mp3
4,400,Joshua Kadison,Jessie,4,3,4,5,4,2,3,2,4,4n8iSiSWRdaSeSpJMbdk9O,NaN,pop,tracks/4n8iSiSWRdaSeSpJMbdk9O.mp3


In [ ]:
# List of the GEMS emotions and their descriptors
emo = ['Wonder (Filled with wonder, Dazzled, Allured, Moved)',
       'Transcendence (Fascinated, Overwhelmed, Feelings of transcendence and spirituality)',
       'Nostalgia (Nostalgic, Dreamy, Sentimental, Melancholic)',
       'Tenderness (Tender, Affectionate, In love, Mellowed)',
       'Peacefulness (Serene, Calm, Soothed, Relaxed)',
       'Joy (Joyful, Amused, Animated, Bouncy)',
       'Sadness (Sad, Sorrowful)',
       'Power (Strong, Triumphant, Energetic, Fiery)',
       'Tension (Tense, Agitated, Nervous, Irritated)']

#### Prediction for all emotions

To get predictions with randomized emotions order uncomment the line 4:<br>
`random.shuffle(emo)`

In [ ]:
for i, row in tqdm(df.iterrows(), total=115):

    # Randomize emotions list
    # random.shuffle(emo) # <- Comment the line for Un-randomized emotion order

    # Prompt definition
    conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "text",
                     "text":f"""Rate the intensity of the following emotions induced by this music excerpt.
Use the following scale : 1 (not at all), 2 (Somewhat), 3 (Moderately), 4 (Quite a lot), 5 (Very Much).
Just give the numeric ratings, without justifying.
- {emo[0]}
- {emo[1]}
- {emo[2]}
- {emo[3]}
- {emo[4]}
- {emo[5]}
- {emo[6]}
- {emo[7]}
- {emo[8]}"""         },
                    {"type": "audio",
                     "path": f'/content/drive/MyDrive/Master/{row['Path']}'},
                ],
            },
        ]

    # Convert conversation into model inputs
    inputs = processor.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
    )
    inputs = inputs.to(device, dtype=torch.bfloat16)
    
    # Generate model response
    outputs = model.generate(**inputs, max_new_tokens=1024)
    decoded_outputs = processor.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # Save predictions to text file
    with open("/MF_finetuned_results.txt", "a+", encoding="utf-8") as f:
        f.write(str(decoded_outputs) + '\n')


#### Single emotion prediction

In [ ]:
for e in emo :
    
    # Retrieve just the emotion name for the output file
    pattern = r'^\w+'
    single_emo = re.findall(pattern, e)[0]
    print(f'Prediction for {single_emo} :')

    for i, row in tqdm(df.iterrows(), total=115):
        
        # Prompt definition
        conversation = [
                {
                    "role": "user",
                    "content": [
                        {"type": "text",
                        "text":f"""Rate the intensity of {e} induced by this music excerpt.
Use the following scale : 1 (not at all), 2 (Somewhat), 3 (Moderately), 4 (Quite a lot), 5 (Very Much).
Just give the numeric rating, without justifying.
"""         },
                        {"type": "audio",
                        "path": f'/content/drive/MyDrive/Master/{row['Path']}'},
                    ],
                },
            ]

        # Convert conversation into model inputs      
        inputs = processor.apply_chat_template(
            conversation,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
        )
        inputs = inputs.to(device, dtype=torch.bfloat16)

        # Generate model response
        outputs = model.generate(**inputs, max_new_tokens=1024)
        decoded_outputs = processor.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)

        # Save predictions to text file  
        with open(f"/MF_{single_emo}_results.txt", "a+", encoding="utf-8") as f:
            f.write(str(decoded_outputs) + '\n')


#### Prediction for the original data score

In [17]:
# Define test dataframe
original_data_path = '../data/test_og.csv'
df = pd.read_csv(original_data_path)
df.head()

,Unnamed: 0,artist,title,Wonder,Transcendence,Nostalgia,Tenderness,Peacefulness,Joy,Power,Tension,Sadness,Spotify ID,Youtube ID,genre,Path
0,457,Sugababes,Push The Button,45.00,14.73,24.91,12.55,12.00,57.36,42.45,17.73,6.91,3EDS89HeAhHiZKlljciK0a,Ugkxu8HzYufqmVieUjWOKqkoQqrrFpFbRnh7,pop,tracks/Ugkxu8HzYufqmVieUjWOKqkoQqrrFpFbRnh7.mp3
1,148,Lacrimas Profundere,The Crown of Leaving,19.29,17.86,17.71,8.57,18.29,12.00,21.07,24.21,17.00,7qNpAPSfPiLQ47cez2IpeW,NaN,rock,tracks/7qNpAPSfPiLQ47cez2IpeW.mp3
2,742,The Cinematic Orchestra,Time and Space,18.29,16.86,9.29,22.00,34.57,2.93,6.93,21.71,31.86,6qqNMdHr1hcyF4amMDP5Sf,NaN,electronic,tracks/6qqNMdHr1hcyF4amMDP5Sf.mp3
3,526,The Beach Boys,Good Vibrations,23.67,8.08,28.25,12.83,27.71,44.50,19.54,6.92,3.04,6aU6a9tdn2vHhnPGlboFZX,UgkxoWtHfVqF7dqNB3LH94zBUkV0DScEs0Gm,pop,tracks/UgkxoWtHfVqF7dqNB3LH94zBUkV0DScEs0Gm.mp3
4,400,Joshua Kadison,Jessie,26.71,17.86,27.50,34.79,30.64,14.29,19.14,8.14,27.43,4n8iSiSWRdaSeSpJMbdk9O,NaN,pop,tracks/4n8iSiSWRdaSeSpJMbdk9O.mp3


In [ ]:
for i, row in tqdm(df.iterrows(), total=115):

    # Prompt definition
    conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "text",
                     "text":f"""Rate the intensity of the following emotions induced by this music excerpt, using a 0-100 scale.
Just give the numeric ratings, without justifying.
- {emo[0]}
- {emo[1]}
- {emo[2]}
- {emo[3]}
- {emo[4]}
- {emo[5]}
- {emo[6]}
- {emo[7]}
- {emo[8]}"""         },
                    {"type": "audio",
                     "path": f'/content/drive/MyDrive/Master/{row['Path']}'},
                ],
            },
        ]

    # Convert conversation into model inputs
    inputs = processor.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
    )
    inputs = inputs.to(device, dtype=torch.bfloat16)

    # Generate model response
    outputs = model.generate(**inputs, max_new_tokens=1024)
    decoded_outputs = processor.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # Save predictions to text file
    with open("/MF_og_results.txt", "a+", encoding="utf-8") as f:
        f.write(str(decoded_outputs) + '\n')